# Helmholtz MHM: complex fields and PML

Published plane/Hankel data and an original transformed PML verification. The small patch below checks complex signs and physical macro balance; the larger studies are loaded from immutable numerical records.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
if not (ROOT / "pixi.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


In [ ]:
from pymhm import TriangleMesh
from pymhm.helmholtz import solve_helmholtz

mesh = TriangleMesh.unit_square(1)
def pressure(p):
    return 1 + (1 + 2j) * p[:, 0] + (2 - 1j) * p[:, 1]
def source(p):
    return -4 * pressure(p)
def impedance(p, normal):
    return normal @ np.array([1 + 2j, 2 - 1j]) - 2j * pressure(p)

with threadpool_limits(1):
    result = solve_helmholtz(mesh, omega=2, source=source, absorbing=impedance,
                             degree=2, local_refinement=2)
    error = result.l2_error(pressure)
    balance = np.max(abs(result.conservation_residuals()))
assert error < 2e-12 and balance < 2e-12
{"pressure_l2_error": error, "macro_balance_max": balance}

## Archived physical norms

The records distinguish polynomial and oscillatory traces, local Qk spaces, pressure and gradient errors. PML uses transformed coefficients and exact Dirichlet data; its approximation error is not an unbounded-domain truncation error.

In [ ]:
record = json.loads((ROOT / "examples/results/helmholtz/comparison.json").read_text())
assert record["source_changed_during_run"] is False
[(r["wave"], r["study"], r["resolution"], r["trace_basis"], r["pressure_relative_error"], r["gradient_relative_error"])
 for r in record["rows"] if r["study"] != "direction"]

In [ ]:
for name in ("convergence", "directions", "plane-oscillatory-fields", "hankel-oscillatory-fields", "pml-convergence", "pml-fields"):
    display(Image(filename=str(ROOT / f"docs/figures/helmholtz/{name}.png")))

The case page states the published data, selected macro/local spaces, nonresonance assumptions, independent DOLFINx/UFL checks and the separate material-data status of the Marmousi benchmark.

## Complete angular sampling

Each published configuration is sampled in all 256 directions. The local Q(ell+2) choice is explicit because the article does not specify a local discretization. The separate local-refinement check does not remove the discrepancy of the absolute ordinates in Figure 6.5.

In [ ]:
article=json.loads((ROOT / "examples/results/helmholtz-article/article.json").read_text())
rows=article["rows"]
assert sum(r["study"]=="direction" for r in rows)==1024
for ell in (2,4):
    for basis in ("polynomial","oscillatory"):
        part=[r for r in rows if r["study"]=="direction" and r["ell"]==ell and r["trace_basis"]==basis]
        assert len(part)==256
        print(ell,basis,max(r["gradient_relative_error"] for r in part))
local = json.loads((ROOT / "examples/results/helmholtz-article/local-refinement-eight.json").read_text())
assert len(local["rows"]) == 4
for row in local["rows"]:
    print("r=8", row["ell"], row["trace_basis"], row["gradient_relative_error"])


In [ ]:
for name in ("article-directions", "article-convergence", "article-local-control"):
    display(Image(filename=str(ROOT / f"docs/figures/helmholtz/{name}.png")))

## Independent full hybrid systems

DOLFINx/UFL independently assembles volume, impedance, incident-wave loads and oriented face couplings for twelve angular cases, then solves the uncondensed complex saddle system with SciPy SuperLU. Both trace families are checked at the two published macro/frequency configurations and three angles. The local Q(ell+2) spaces remain the explicitly selected finite discretization. These field and norm comparisons verify that discretization; they do not remove the discrepancies with the historical plotted ordinates.


In [ ]:
native = json.loads((ROOT / "examples/results/helmholtz-article/native-global-verification.json").read_text())
assert native["source_changed_during_run"] is False
assert len(native["rows"]) == 12
for row in native["rows"]:
    assert row["native_original_equation_relative_residual"] < 1e-10
    assert row["pressure_coefficient_max_difference"] < 1e-8
    assert abs(row["native_relative_pressure_l2"] - row["pymhm_relative_pressure_l2"]) < 1e-9
    assert abs(row["native_relative_gradient_l2"] - row["pymhm_relative_gradient_l2"]) < 1e-9
native["verification"]


## Pollution and projected exact flux

For the outgoing Hankel solution, the local interpolant lifts the facewise L2 projection of the exact normal flux using the same local inverse as MHM. The ratio of the two broken gradient errors is compared with the published factor three. Constant traces use relative errors and linear traces absolute errors; that common normalization does not change the ratio.

The local Q(ell+3) spaces and two subdivisions per direction are the selected finite local discretization. A reported threshold applies to the recorded admissible sequence. It is not a stability guarantee at every unsampled mesh size; rejected resonant local inverses interrupt the admissible suffix.


In [ ]:
from examples.helmholtz_threshold import sampled_threshold

stability_records = []
for ell, frequency in ((0, 10), (0, 20), (1, 15), (1, 75)):
    path = ROOT / f"examples/results/helmholtz-stability/ell{ell}-frequency{frequency}.json"
    sequence = json.loads(path.read_text())
    rows = sequence["rows"]
    assert all(np.isfinite(row["ratio"]) and row["ratio"] >= 0 for row in rows)
    assessment = sampled_threshold(rows, sequence.get("excluded_settings", ()))
    stability_records.append({
        "ell": ell, "frequency": frequency, "levels": len(rows),
        "finest_n": rows[-1]["n"], "finest_ratio": rows[-1]["ratio"],
        **assessment,
    })
stability_records


In [ ]:
for name in ("stability-errors", "stability-ratio"):
    display(Image(filename=str(ROOT / f"docs/figures/helmholtz/{name}.png")))
